[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BASE-Laboratory/OpenImpala/blob/master/notebooks/profiling_and_tuning.ipynb)

# OpenImpala Profiling & Bottleneck Hunt

This notebook exists for **one job**: find out where OpenImpala is spending
its time, and whether the slowdown is in Python bindings, AMReX/HYPRE setup,
or the linear solve itself.

Each `oi.tortuosity(numpy_array, ...)` call rebuilds the entire AMReX +
HYPRE stack from scratch and runs a percolation flood-fill before the solve.
That per-call overhead can dominate for small/medium problems. We isolate it
in the first four sections, *then* run the deeper C++/GPU profilers.

**Sections**

1. Setup
2. Synthetic datasets (tiny → medium)
3. **Python-level stage breakdown** — time every stage of one solve
4. **Fixed-overhead extraction** — fit `t(N) = a + b·Nᵖ` across sizes
5. **Amortization test** — reuse VoxelImage across repeated solves
6. **cProfile** — Python-level hottest frames
7. **AMReX TinyProfiler** — C++ function breakdown
8. **Nsight Systems** — GPU kernel timeline (if GPU present)
9. Solver comparison (focused, post-diagnosis)
10. `max_grid_size` tuning (focused, post-diagnosis)
11. Native-binary comparison *(optional, isolates binding overhead)*
12. Summary & HPC recommendations

If you only have 5 minutes, run Sections 1–6. Sections 7+ need a real GPU
and a built `openimpala` executable for full value.


## 1. Setup

Detect hardware, install dependencies, import, start a session.


In [ ]:
import subprocess
try:
    r = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                       capture_output=True, text=True, timeout=10)
    gpu_name = r.stdout.strip()
    print(f"GPU detected: {gpu_name}" if gpu_name else "No GPU — CPU only.")
except FileNotFoundError:
    gpu_name = ""
    print("nvidia-smi not found — CPU only.")


In [ ]:
!apt-get install -y libopenmpi-dev > /dev/null 2>&1
!pip install openimpala porespy py-spy > /dev/null 2>&1
print("Dependencies installed.")


In [ ]:
import openimpala as oi
from openimpala import core as oic  # noqa: F401
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time, io, sys, re, cProfile, pstats, warnings

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
print(f"OpenImpala version: {oi.__version__}")


In [ ]:
session = oi.Session()
session.__enter__()
print("OpenImpala session started.")


## 1a. Is the openimpala build actually using the GPU?

Most "OpenImpala is slow on Colab" reports turn out to be the pip wheel
being **CPU-only + OpenMP** while a perfectly good T4 sits idle. We
probe the build by running a trivial solve in a subprocess with
`verbose=1` and inspecting the init banner.


In [ ]:
# Use oi.build_info() when the wheel provides it (4.0.2+). Older wheels
# fall back to the subprocess banner probe so the notebook still works.
backend = None
info = None
try:
    info = oi.build_info()
    backend = info["backend"]  # "cpp-cuda", "cpp-hip", "cpp-cpu", "pure-python"
    print(f"openimpala backend: {backend}")
    print(f"  version:        {info.get('version', 'unknown')}")
    print(f"  OpenMP threads: {info.get('openmp_max_threads', 1)}")
    print(f"  MPI enabled:    {info.get('mpi_enabled', False)}")
    print(f"  HYPRE CUDA:     {info.get('hypre_cuda', False)}")
    print(f"  TinyProfile:    {info.get('tiny_profile', False)}")
    if info.get("gpu_device_count", -1) > 0:
        print(f"  GPU devices:    {info['gpu_device_count']}")
except AttributeError:
    # Pre-4.0.2 wheel — no build_info(). Fall back to banner probe.
    import subprocess, tempfile, os, textwrap
    probe = textwrap.dedent("""
        import numpy as np, openimpala as oi
        arr = np.zeros((8, 8, 8), dtype=np.int32)
        with oi.Session():
            try: oi.tortuosity(arr, phase=0, direction='z', solver='pcg', verbose=1)
            except Exception as e: print(f'__ERR__ {e}')
    """).strip()
    tmp = tempfile.mkdtemp()
    path = os.path.join(tmp, "probe.py")
    with open(path, "w") as f:
        f.write(probe)
    r = subprocess.run([sys.executable, path], capture_output=True, text=True, timeout=90)
    banner = (r.stdout or "") + (r.stderr or "")
    low = banner.lower()
    if "cuda" in low or ("gpu" in low and "no gpu" not in low):
        backend = "cpp-cuda"
    elif "hip" in low and "rocm" in low:
        backend = "cpp-hip"
    else:
        backend = "cpp-cpu"
    print(f"openimpala backend (banner probe): {backend}")
    print("(install openimpala 4.0.2+ for a proper build_info() API)")

is_gpu_build = backend in ("cpp-cuda", "cpp-hip")

if gpu_name and not is_gpu_build:
    print()
    print("=" * 70)
    print("  ⚠  CPU-ONLY BUILD DETECTED ON A GPU-EQUIPPED HOST")
    print("=" * 70)
    print(f"  Host GPU:          {gpu_name}")
    print(f"  openimpala build:  {backend}")
    print()
    print("  Your GPU is idle. Every solve in this notebook is CPU-bound.")
    print("  For single-phase 3D diffusion on 128^3+ grids, a CUDA build is")
    print("  typically 10-50x faster — this is the dominant source of")
    print("  'OpenImpala feels slow on Colab'.")
    print()
    print("  Fix: install the CUDA wheel instead.")
    print("    !pip uninstall -y openimpala")
    print("    !pip install openimpala-cuda")
    print("  Then Runtime -> Restart session and re-run from the top.")
    print("=" * 70)
elif is_gpu_build:
    print(f"\n✓ {backend.upper()} build active on {gpu_name or 'accelerator'}")
else:
    print("\nCPU-only run (no accelerator detected).")


## 2. Synthetic datasets

Small sizes chosen so a full profiling run completes in minutes, not hours.
`data_micro` (16³) is tiny enough that its solve time is ~all overhead — we
use it to extract the per-call fixed cost.


In [ ]:
import porespy as ps

def make_micro(size, porosity=0.5, seed=42):
    np.random.seed(seed)
    im = ps.generators.blobs(shape=[size] * 3, porosity=porosity, blobiness=1.5)
    return im.astype(np.int32)

sizes = [16, 32, 64, 96, 128]
datasets = {n: make_micro(n) for n in sizes}
data_medium = datasets[64]  # default workhorse for most diagnostics

for n, d in datasets.items():
    print(f"  {n:>4d}^3  porosity={np.mean(d == 0):.3f}  bytes={d.nbytes/1e6:.2f} MB")


## 3. Python-level stage breakdown — one call, one bar chart

`oi.tortuosity(numpy_array, ...)` is not a single operation. From
`facade.py` it is roughly:

```
numpy → VoxelImage.from_numpy    (copy + AMReX BoxArray/Geometry/iMultiFab build)
PercolationCheck(img, ...)        (flood fill — runs every call!)
VolumeFraction(img, ...)          (cheap count)
TortuosityHypre(img, ..., solver) (HYPRE grid/stencil/matrix assembly)
solver.value()                    (actual linear solve + flux integration)
```

We time each stage independently. Whichever bar dominates *is* the
bottleneck. If `from_numpy` or `TortuosityHypre(...)` construction dominates,
your slowdown is in binding/setup — not the solver.


In [ ]:
from openimpala import _core as _core
from openimpala.facade import _parse_direction, _parse_solver

def time_stages(data, solver="pcg", direction="z", max_grid_size=32):
    """Decompose a single tortuosity call into timed binding stages.

    `_numpy_to_voxelimage` is split into:
      - numpy ascontiguousarray copy (pure Python / numpy)
      - VoxelImage.from_numpy (pybind11 + AMReX grid build + voxel write)
    so you can tell if the cost is the copy or the AMReX side.
    """
    d = _parse_direction(direction)
    st = _parse_solver(solver)
    out = {}

    t0 = time.perf_counter()
    arr = np.ascontiguousarray(data, dtype=np.int32)
    out["np.ascontiguousarray copy"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    img = _core.VoxelImage.from_numpy(arr, max_grid_size)
    out["VoxelImage.from_numpy (AMReX grid + voxel write)"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    pc = _core.PercolationCheck(img, 0, d, 0)
    out["PercolationCheck (flood fill)"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    vf_val = _core.VolumeFraction(img, 0, 0).value_vf()
    out["VolumeFraction"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    solver_obj = _core.TortuosityHypre(img, vf_val, 0, d, st, ".", 0.0, 1.0, 0, False)
    out["TortuosityHypre ctor (HYPRE setup + matrix assembly)"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    tau = solver_obj.value()
    out["solver.value() (solve + flux)"] = time.perf_counter() - t0

    out["_total"] = sum(v for k, v in out.items() if not k.startswith("_"))
    out["_tau"] = tau
    out["_iters"] = solver_obj.iterations
    return out

# Warm-up — first call pays lazy-init inside AMReX/HYPRE
_ = time_stages(datasets[32])

stages = time_stages(data_medium, solver="pcg")
print(f"tau = {stages['_tau']:.4f}   iters = {stages['_iters']}   total = {stages['_total']:.3f}s\n")
for k, v in stages.items():
    if k.startswith("_"):
        continue
    print(f"  {k:55s}  {v:7.3f}s  ({100*v/stages['_total']:5.1f}%)")


In [ ]:
stage_names = [k for k in stages if not k.startswith("_")]
stage_times = [stages[k] for k in stage_names]

fig, ax = plt.subplots(figsize=(11, 4))
y = np.arange(len(stage_names))
colors = ["#9b59b6", "#8e44ad", "#2ecc71", "#95a5a6", "#f39c12", "#e74c3c"]
ax.barh(y, stage_times, color=colors[:len(stage_names)], alpha=0.85, edgecolor="white")
for i, t in enumerate(stage_times):
    ax.text(t + max(stage_times) * 0.01, i,
            f"{t:.3f}s  ({100*t/stages['_total']:.1f}%)", va="center", fontsize=9)
ax.set_yticks(y)
ax.set_yticklabels(stage_names, fontsize=9)
ax.set_xlabel("Wall time (s)")
ax.set_title("Single oi.tortuosity(64³) call — stage breakdown", fontweight="bold")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

slowest = max(stage_names, key=lambda k: stages[k])
print(f"\nBottleneck at 64³: {slowest} ({100*stages[slowest]/stages['_total']:.1f}%)")


### §3b Per-stage scaling — does this stage scale with N, or is it fixed overhead?

Run the same decomposition at 32³ and 128³ and plot the ratio per stage.
A stage with ratio ≈ `(128/32)³ = 64×` is doing O(N) work — legitimate
compute. A stage with ratio ≈ 1× is constant overhead — every call pays
the same price regardless of problem size, which is the *classic* signature
of binding / setup cost.


In [ ]:
stages_small = time_stages(datasets[32])
stages_big   = time_stages(datasets[128])

# Ideal O(N) ratio when scaling voxel count by (128/32)^3
N_ratio = (128 / 32) ** 3
print(f"Ideal O(N) work ratio for 32->128: {N_ratio:.0f}×")
print(f"{'stage':<55s} {'32³ (s)':>10s} {'128³ (s)':>10s} {'ratio':>8s}  behaviour")
print("-" * 110)

rows = []
for k in stage_names:
    ts, tb = stages_small[k], stages_big[k]
    r = tb / max(ts, 1e-9)
    if r > 0.5 * N_ratio:
        label = "scales O(N) ✓"
    elif r < 2.0:
        label = "~constant (overhead)"
    else:
        label = "sub-linear"
    rows.append({"stage": k, "t_32": ts, "t_128": tb, "ratio": r, "label": label})
    print(f"{k:<55s} {ts:>10.3f} {tb:>10.3f} {r:>8.1f}×  {label}")

df_scale = pd.DataFrame(rows)


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
y = np.arange(len(stage_names))
w = 0.4
ax.barh(y - w/2, df_scale["t_32"], w, color="#3498db", alpha=0.85, label="32³")
ax.barh(y + w/2, df_scale["t_128"], w, color="#e74c3c", alpha=0.85, label="128³")
for i, r in enumerate(df_scale["ratio"]):
    ax.text(df_scale["t_128"].iloc[i] * 1.02, i + w/2, f"{r:.0f}×",
            va="center", fontsize=8, color="#2c3e50")
ax.set_yticks(y)
ax.set_yticklabels(stage_names, fontsize=9)
ax.set_xlabel("Wall time (s)")
ax.set_title(f"Per-stage scaling (ideal O(N) ratio = {N_ratio:.0f}×)", fontweight="bold")
ax.axvline(0, color="#bdc3c7", linewidth=0.5)
ax.invert_yaxis()
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

constant_stages = df_scale[df_scale["ratio"] < 2.0]
if not constant_stages.empty:
    print("\n⚠ Stages that DON'T scale with problem size (pure overhead):")
    for _, r in constant_stages.iterrows():
        print(f"    {r['stage']:55s}  ratio={r['ratio']:.1f}×")
    print("  Every oi.tortuosity() call pays these, regardless of image size.")


## 4. Fixed-overhead extraction

A call's wall time has two parts: a fixed per-call overhead `a` (binding
dispatch, AMReX/HYPRE setup, flood fill) and a size-dependent compute part
`b·Nᵖ`. Fit `t(N) = a + b·Nᵖ` across sizes.

If `a` is a meaningful fraction of the medium-size solve time, every solve
you run is paying a tax that has nothing to do with the actual physics.


In [ ]:
overhead_rows = []
for n, d in datasets.items():
    # Two trials — median to reduce jitter without paying for 3×
    ts = []
    for _ in range(2):
        t0 = time.perf_counter()
        oi.tortuosity(d, phase=0, direction="z", solver="pcg",
                      max_grid_size=32, verbose=0)
        ts.append(time.perf_counter() - t0)
    overhead_rows.append({"n": n, "voxels": n**3, "t": float(np.median(ts))})
    print(f"  {n:>4d}^3  {overhead_rows[-1]['t']:.3f}s")

df_oh = pd.DataFrame(overhead_rows)
df_oh


In [ ]:
from scipy.optimize import curve_fit

N = df_oh["voxels"].values.astype(float)
T = df_oh["t"].values

def model(x, a, b, p):
    return a + b * np.power(x, p)

try:
    popt, _ = curve_fit(model, N, T, p0=[T.min(), 1e-9, 1.0], maxfev=20000)
    a_raw, b_fit, p_fit = popt
except Exception as e:
    print(f"Fit failed: {e}")
    a_raw, b_fit, p_fit = T.min(), 0.0, 1.0

# Clamp: a negative intercept means there is no meaningful fixed overhead —
# pretending otherwise misleads the summary.
a_fit = max(a_raw, 0.0)
overhead_meaningful = a_fit > 0.01 * T.max()

df_oh["overhead_pct"] = 100 * a_fit / df_oh["t"]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.loglog(N, T, "o", color="#2c3e50", markersize=10, label="Measured")
xs = np.logspace(np.log10(N.min()), np.log10(N.max()), 200)
ax.loglog(xs, model(xs, a_fit, b_fit, p_fit), "-", color="#3498db",
          label=f"Fit: {a_fit:.2f}s + {b_fit:.2e}·N^{p_fit:.2f}")
if overhead_meaningful:
    ax.axhline(a_fit, color="#e74c3c", linestyle="--",
               label=f"Fixed overhead ≈ {a_fit:.2f}s")
ax.set_xlabel("Voxels (N)")
ax.set_ylabel("Wall time (s)")
ax.set_title("Fixed overhead vs compute", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

if a_raw < 0:
    print(f"\nFit intercept was negative ({a_raw:.2f}s) — clamped to 0.")
    print("Interpretation: no detectable per-call fixed overhead; all time is compute.")
print(f"Estimated fixed overhead per call: {a_fit:.2f}s")
print(f"Compute scaling:                  O(N^{p_fit:.2f})")

if p_fit > 1.2:
    print(f"\n⚠ Super-linear compute scaling (O(N^{p_fit:.2f}) vs ideal O(N)).")
    print("  Likely cause: Krylov iteration count growing with N. A multigrid")
    print("  preconditioner (SMG/PFMG) or MLMG would bring this closer to O(N).")

print("\nOverhead share per size:")
print(df_oh[["n", "t", "overhead_pct"]].to_string(index=False))


## 5. Amortization test — reuse VoxelImage across calls

If most time is spent rebuilding AMReX/HYPRE state per call, then calling
the solver several times on the **same** `VoxelImage` should get cheaper
after the first call (matrix is already assembled inside the solver).

We bypass `oi.tortuosity` and hit `_core.TortuosityHypre` directly so the
`VoxelImage` is reused. Compare this to calling `oi.tortuosity(numpy, ...)`
five times — each rebuild pays the full overhead.


In [ ]:
from openimpala.facade import _numpy_to_voxelimage, _parse_direction, _parse_solver

def naive_repeat(data, n=5, solver="pcg"):
    """Top-level facade — rebuilds everything every call."""
    ts = []
    for _ in range(n):
        t0 = time.perf_counter()
        oi.tortuosity(data, phase=0, direction="z", solver=solver,
                      max_grid_size=32, verbose=0)
        ts.append(time.perf_counter() - t0)
    return ts

def amortized_repeat(data, n=5, solver="pcg"):
    """Build VoxelImage once, reuse across solves."""
    d = _parse_direction("z")
    st = _parse_solver(solver)
    img = _numpy_to_voxelimage(data, 32)
    vf = _core.VolumeFraction(img, 0, 0).value_vf()

    ts = []
    for _ in range(n):
        t0 = time.perf_counter()
        s = _core.TortuosityHypre(img, vf, 0, d, st, ".", 0.0, 1.0, 0, False)
        s.value()
        ts.append(time.perf_counter() - t0)
    return ts

n_repeat = 5
naive = naive_repeat(data_medium, n_repeat)
amort = amortized_repeat(data_medium, n_repeat)

print(f"Naive  oi.tortuosity(numpy) × {n_repeat}: " + ", ".join(f"{t:.2f}s" for t in naive))
print(f"Amortized (reuse VoxelImage): " + ", ".join(f"{t:.2f}s" for t in amort))
print(f"\nNaive total:      {sum(naive):.2f}s  (mean {np.mean(naive):.2f}s/call)")
print(f"Amortized total:  {sum(amort):.2f}s  (mean {np.mean(amort):.2f}s/call)")
print(f"Speedup from reuse: {sum(naive)/max(sum(amort), 1e-9):.2f}×")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(1, n_repeat + 1)
ax.plot(x, naive, "o-", color="#e74c3c", linewidth=2, markersize=9,
        label="Naive: oi.tortuosity(numpy, ...)")
ax.plot(x, amort, "o-", color="#2ecc71", linewidth=2, markersize=9,
        label="Amortized: reuse VoxelImage + TortuosityHypre")
ax.set_xlabel("Call #")
ax.set_ylabel("Wall time (s)")
ax.set_title("Per-call cost: rebuild-everything vs reuse", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

gap = np.mean(naive) - np.mean(amort)
print(f"Average per-call rebuild tax: {gap:.2f}s "
      f"({100*gap/np.mean(naive):.0f}% of a naive call)")


## 6. cProfile — Python-level hottest frames

`cProfile` shows cumulative time in Python frames, including time spent
inside pybind11 wrappers. Look for big chunks in:

- `_numpy_to_voxelimage` or `VoxelImage.from_numpy` → data copy cost
- `PercolationCheck.__init__` → flood-fill dominates
- `TortuosityHypre.__init__` → matrix assembly dominates
- `TortuosityHypre.value` → the actual solve

Anything above these (e.g. `_parse_direction`, `_parse_solver`) appearing
in the top 20 indicates pure Python dispatch waste.


In [ ]:
pr = cProfile.Profile()
pr.enable()
oi.tortuosity(data_medium, phase=0, direction="z", solver="pcg",
              max_grid_size=32, verbose=0)
pr.disable()

stream = io.StringIO()
ps_stats = pstats.Stats(pr, stream=stream).sort_stats("cumulative")
ps_stats.print_stats(20)
print(stream.getvalue())


## 7. AMReX TinyProfiler — C++ internal breakdown

Now we go *below* the binding layer. AMReX's TinyProfiler instruments
every `BL_PROFILE`-annotated function in the C++ source and prints a table
when the session finalizes.

This is especially important given what §3 just showed us: `solver.value()`
is ~96% of wall time at 64³ and scales super-linearly to 128³. TinyProfiler
will break that down into `TortuosityHypre::solve`, `setupMatrixEquation`,
`preconditionPhaseFab`, `global_fluxes`, `computePlaneFluxes`, etc.

**Implementation note**: TinyProfiler prints via C++ streams on AMReX
finalize, which Python's `sys.stdout` redirection cannot capture (and
recycling a Session inside the same process is fragile). We run the solve
in a short-lived **subprocess** — Python's `capture_output=True` grabs
fd 1 + fd 2 at the OS level, so we get everything.


In [ ]:
import subprocess, tempfile, os, textwrap, json as _json

def run_with_profiler(data, solver="pcg", direction="z", max_grid_size=32, timeout=600):
    """Run ONE profiled solve in a subprocess.

    Why a subprocess: TinyProfiler prints via C++ streams on AMReX finalize,
    which sys.stdout redirection can't capture. A subprocess gives us a clean
    AMReX lifecycle and Python's ``capture_output=True`` grabs fd 1 + fd 2
    at the OS level — including everything TinyProfiler emits.

    Returns (result_dict, combined_stdout_stderr).
    """
    tmp = tempfile.mkdtemp()
    npy_path = os.path.join(tmp, "data.npy")
    np.save(npy_path, data)

    script = textwrap.dedent(f"""
        import json, numpy as np, openimpala as oi
        data = np.load({npy_path!r})
        with oi.Session():
            r = oi.tortuosity(data, phase=0, direction={direction!r},
                              solver={solver!r}, max_grid_size={max_grid_size},
                              verbose=1)
            print("__RESULT__" + json.dumps({{
                "tau": float(r.tortuosity),
                "converged": bool(r.solver_converged),
                "iters": int(r.iterations),
                "residual": float(r.residual_norm),
            }}))
    """).strip()
    script_path = os.path.join(tmp, "run.py")
    with open(script_path, "w") as f:
        f.write(script)

    proc = subprocess.run([sys.executable, script_path],
                          capture_output=True, text=True, timeout=timeout)
    combined = (proc.stdout or "") + "\n" + (proc.stderr or "")

    # Pull the result line out of stdout
    result = None
    for line in combined.splitlines():
        if line.startswith("__RESULT__"):
            try:
                result = _json.loads(line[len("__RESULT__"):])
            except Exception:
                pass
            break

    if proc.returncode != 0:
        print(f"⚠ subprocess exited with code {proc.returncode}")
    return result, combined

print("Running a profiled solve on 64^3 in a subprocess…")
prof_result, prof_output = run_with_profiler(data_medium, solver="pcg")
if prof_result is not None:
    print(f"tau={prof_result['tau']:.4f}  converged={prof_result['converged']}  iters={prof_result['iters']}")
else:
    print("No result line found — solve may have failed; check output below.")
print(f"Captured output length: {len(prof_output)} chars")

# Show the tail (TinyProfiler dumps near the end, on finalize)
if prof_output.strip():
    print("\n--- Last 4000 chars of captured output ---")
    print(prof_output[-4000:])
else:
    print("\n(No output captured.)")


In [ ]:
def parse_tiny_profiler(output):
    """Parse AMReX TinyProfiler output into a DataFrame."""
    records = []
    pattern = re.compile(
        r"^\s*(\S+.*\S)\s+"
        r"(\d+)\s+"
        r"([\d.e+-]+)\s+"
        r"([\d.e+-]+)\s+"
        r"([\d.e+-]+)\s+"
        r"([\d.e+-]+)\s*%",
        re.MULTILINE,
    )
    for m in pattern.finditer(output):
        records.append({
            "function": m.group(1).strip(),
            "ncalls": int(m.group(2)),
            "excl_min_s": float(m.group(3)),
            "excl_avg_s": float(m.group(4)),
            "excl_max_s": float(m.group(5)),
            "max_pct": float(m.group(6)),
        })
    return pd.DataFrame(records)

df_prof = parse_tiny_profiler(prof_output)

# Heuristic: TinyProfiler tables always contain "TinyProfiler" or
# "Name   NCalls" header. If neither is present, the build just doesn't
# emit a table and there is nothing to parse.
has_tp_header = ("TinyProfiler" in prof_output
                 or re.search(r"Name\s+NCalls", prof_output) is not None)

if len(df_prof) > 0:
    df_prof = df_prof.sort_values("excl_avg_s", ascending=False).head(15)
    print(f"Top {len(df_prof)} C++ functions by exclusive time:")
    print(df_prof.to_string(index=False))
elif not has_tp_header:
    print("=" * 70)
    print("  ⚠  No TinyProfiler table was emitted by this openimpala build")
    print("=" * 70)
    print("  The subprocess ran to completion (result captured above) but")
    print("  AMReX did not dump a TinyProfiler table on finalize. This means")
    print("  the C++ side was built without the profiling flag.")
    print()
    print("  To enable, rebuild openimpala's AMReX dependency with one of:")
    print("    CMake:  -DAMReX_TINY_PROFILE=ON")
    print("    GNUmake: TINY_PROFILE=TRUE")
    print()
    print("  Until then §7 cannot break down the C++ solve into components.")
    print("  §3 (Python stage breakdown) + §3b (per-stage scaling) already")
    print("  show that solver.value() is ~85-95% of wall time and scales")
    print("  super-linearly — that IS the bottleneck regardless.")
    print("=" * 70)
else:
    print("TinyProfiler header detected but no rows parsed — regex mismatch.")
    print("Dump tail of raw output above and adjust parse_tiny_profiler().")


In [ ]:
if len(df_prof) > 0 and "max_pct" in df_prof.columns:
    def categorize(name):
        n = name.lower()
        if "solve" in n:
            return "#e74c3c"
        if any(k in n for k in ["setup", "fill", "matrix", "stencil", "assemble"]):
            return "#f39c12"
        if any(k in n for k in ["flux", "plane"]):
            return "#3498db"
        if any(k in n for k in ["mask", "precondition", "flood", "percolation"]):
            return "#2ecc71"
        return "#95a5a6"

    colors = [categorize(f) for f in df_prof["function"]]
    y = np.arange(len(df_prof))

    fig, ax = plt.subplots(figsize=(10, max(4, len(df_prof) * 0.4)))
    ax.barh(y, df_prof["excl_avg_s"], color=colors, alpha=0.85, edgecolor="white")
    for i, (_, row) in enumerate(df_prof.iterrows()):
        ax.text(row["excl_avg_s"] + 0.001, i, f'{row["max_pct"]:.1f}%',
                va="center", fontsize=9)
    ax.set_yticks(y)
    ax.set_yticklabels(df_prof["function"], fontsize=9)
    ax.set_xlabel("Exclusive time (s)")
    ax.set_title("AMReX TinyProfiler — C++ function breakdown", fontweight="bold")
    ax.invert_yaxis()

    from matplotlib.patches import Patch
    legend = [
        Patch(facecolor="#e74c3c", label="Linear solve"),
        Patch(facecolor="#f39c12", label="Matrix assembly/setup"),
        Patch(facecolor="#3498db", label="Flux computation"),
        Patch(facecolor="#2ecc71", label="Pre-processing"),
        Patch(facecolor="#95a5a6", label="Other"),
    ]
    ax.legend(handles=legend, loc="lower right", framealpha=0.9)
    plt.tight_layout()
    plt.show()
else:
    print("Skipping chart — no TinyProfiler data.")


## 8. NVIDIA Nsight Systems — GPU kernel timeline

If a GPU is available, `nsys` captures a kernel-level timeline: launches,
memory transfers, idle gaps. Signatures of Python-binding overhead on the
GPU path are:

- Many tiny launches with gaps → Python loop driving the GPU
- Frequent H2D transfers of the same data → `numpy → VoxelImage` copies leaking out
- Long idle gaps between kernels → CPU-side serial work between solver calls

We use a standalone script so the subprocess is clean.


In [ ]:
import shutil
nsys_available = shutil.which("nsys") is not None and bool(gpu_name)

if not nsys_available:
    if not gpu_name:
        print("No GPU — skipping Nsight. (Runtime > Change runtime type > T4 GPU)")
    else:
        print("nsys not found — install via: apt-get install nsight-systems")
else:
    print(f"nsys: {shutil.which('nsys')}   GPU: {gpu_name}")
    script = '''
import openimpala as oi, numpy as np, porespy as ps
np.random.seed(42)
data = ps.generators.blobs(shape=[64]*3, porosity=0.5, blobiness=1.5).astype(np.int32)
with oi.Session():
    r = oi.tortuosity(data, phase=0, direction="z", solver="pcg",
                      max_grid_size=32, verbose=0)
    print(f"tau={r.tortuosity:.4f} converged={r.solver_converged}")
'''
    with open("/tmp/oi_profile.py", "w") as f:
        f.write(script)
    !nsys profile --output=/tmp/oi_profile --force-overwrite=true \
        --trace=cuda,nvtx,osrt --cuda-memory-usage=true --stats=true \
        python3 /tmp/oi_profile.py 2>&1 | tail -60
    print("\nReport saved to /tmp/oi_profile.nsys-rep (download for Nsight GUI).")


In [ ]:
if nsys_available:
    stats = subprocess.run(
        ["nsys", "stats", "--report", "cuda_gpu_kern_sum",
         "--format", "csv", "/tmp/oi_profile.nsys-rep"],
        capture_output=True, text=True, timeout=60,
    )
    if stats.returncode == 0 and stats.stdout.strip():
        lines = stats.stdout.strip().split("\n")
        hdr = next((i for i, l in enumerate(lines) if "Time" in l and "Name" in l), None)
        if hdr is not None:
            df_k = pd.read_csv(io.StringIO("\n".join(lines[hdr:])))
            if "Total Time (ns)" in df_k.columns:
                df_k["Total Time (ms)"] = df_k["Total Time (ns)"] / 1e6
                df_k = df_k.sort_values("Total Time (ns)", ascending=False).head(10)
                print("Top CUDA kernels by total GPU time:")
                print(df_k[["Name", "Total Time (ms)", "Instances"]].to_string(index=False))
            else:
                print(df_k.head().to_string(index=False))
        else:
            print(stats.stdout[:2000])
    else:
        print("nsys stats produced no kernel data.")


In [ ]:
if nsys_available:
    mem = subprocess.run(
        ["nsys", "stats", "--report", "cuda_gpu_mem_size_sum",
         "--format", "csv", "/tmp/oi_profile.nsys-rep"],
        capture_output=True, text=True, timeout=60,
    )
    if mem.returncode == 0 and mem.stdout.strip():
        lines = mem.stdout.strip().split("\n")
        hdr = next((i for i, l in enumerate(lines)
                    if "Operation" in l or "Total" in l), None)
        if hdr is not None:
            df_mem = pd.read_csv(io.StringIO("\n".join(lines[hdr:])))
            print("CUDA memory transfer summary (H2D/D2H):")
            print(df_mem.to_string(index=False))
            print("\nRed flags:")
            print("  • Frequent H2D transfers of similar size → numpy→VoxelImage")
            print("    copy leaking out of bindings into the hot path")
            print("  • Large D2H transfer per call → result readback overhead")
        else:
            print(mem.stdout[:1500])
    else:
        print("nsys mem report produced no data.")


## 9. Solver comparison *(focused, post-diagnosis)*

Now that you know where time goes, compare HYPRE solvers on `data_medium`
at a single grid size. `n_repeats=1` — this sweep is for picking a solver,
not measuring steady-state noise.


In [ ]:
# Compare Krylov and multigrid choices on 64^3.
# Each entry is (label, solver, preconditioner). A preconditioner of None means
# the solver ignores it (standalone SMG/PFMG/Jacobi, or MLMG). The PCG+PFMG and
# GMRES+PFMG rows are the textbook-best combinations for Poisson-like problems
# and the ones most likely to win as N grows.
combos = [
    ("pcg",          "pcg",       None),
    ("pcg+pfmg",     "pcg",       "pfmg"),
    ("pcg+smg",      "pcg",       "smg"),
    ("flexgmres+pfmg", "flexgmres", "pfmg"),
    ("gmres",        "gmres",     None),
    ("gmres+pfmg",   "gmres",     "pfmg"),
    ("bicgstab",     "bicgstab",  None),
    ("smg",          "smg",       None),
    ("pfmg",         "pfmg",      None),
    ("mlmg",         "mlmg",      None),
]

records = []
for label, s, pc in combos:
    kwargs = dict(phase=0, direction="z", solver=s, max_grid_size=32, verbose=0)
    if pc is not None:
        kwargs["preconditioner"] = pc
    try:
        t0 = time.perf_counter()
        res = oi.tortuosity(data_medium, **kwargs)
        dt = time.perf_counter() - t0
        records.append({"solver": label, "t": dt, "iters": res.iterations,
                        "tau": res.tortuosity, "ok": res.solver_converged})
        print(f"  {label:18s}  t={dt:6.2f}s  iters={res.iterations:4d}  tau={res.tortuosity:.4f}")
    except TypeError as e:
        # Older wheels don't know the 'preconditioner' kwarg — fall back without it.
        if "preconditioner" in str(e) and pc is not None:
            print(f"  {label:18s}  SKIP — wheel predates preconditioner plumbing")
            records.append({"solver": label, "t": np.nan, "iters": np.nan,
                            "tau": np.nan, "ok": False})
            continue
        raise
    except Exception as e:
        records.append({"solver": label, "t": np.nan, "iters": np.nan,
                        "tau": np.nan, "ok": False})
        print(f"  {label:18s}  FAILED — {e}")

df_solvers = pd.DataFrame(records)
df_solvers


In [ ]:
ok = df_solvers.dropna(subset=["t"])
best_solver = ok.loc[ok["t"].idxmin(), "solver"] if not ok.empty else "pcg"

fig, ax1 = plt.subplots(figsize=(11, max(4, 0.5 * len(df_solvers))))
colors = ["#2ecc71" if c else "#e74c3c" for c in df_solvers["ok"]]
y = np.arange(len(df_solvers))

ax1.barh(y, df_solvers["t"], color=colors, alpha=0.85, edgecolor="white")
ax1.set_yticks(y)
ax1.set_yticklabels(df_solvers["solver"])
ax1.set_xlabel("Wall time (s)", color="#2c3e50")
ax1.set_title(f"Solver comparison — 64³, best by wall time: {best_solver}",
              fontweight="bold")
ax1.invert_yaxis()

# Overlay iteration counts on a twin axis — this is the key diagnostic.
# A solver with few iterations but expensive each (multigrid) can still
# win overall; wall time alone hides that.
ax2 = ax1.twiny()
ax2.plot(df_solvers["iters"], y, "D", color="#8e44ad", markersize=9,
         markeredgecolor="white", markeredgewidth=1.2, zorder=5)
ax2.set_xlabel("Iterations", color="#8e44ad")
ax2.tick_params(axis="x", colors="#8e44ad")

# Annotate each row with iter count and per-iter cost
for i, (_, row) in enumerate(df_solvers.iterrows()):
    if row["ok"] and not np.isnan(row["t"]):
        per_iter = row["t"] / max(row["iters"], 1)
        ax1.text(row["t"] * 1.02, i,
                 f" {int(row['iters'])} iters  ({per_iter*1000:.0f} ms/iter)",
                 va="center", fontsize=8, color="#2c3e50")

from matplotlib.patches import Patch
legend = [
    Patch(facecolor="#2ecc71", label="Converged"),
    Patch(facecolor="#e74c3c", label="Failed"),
    plt.Line2D([0], [0], marker="D", color="#8e44ad", linestyle="None",
               markersize=8, label="Iterations"),
]
ax1.legend(handles=legend, loc="lower right", framealpha=0.9)
plt.tight_layout()
plt.show()

# Flag solvers with few iterations — likely multigrid/preconditioned.
# If one converged in <10 iterations but is slower per iteration, it may
# still be the better choice at larger problem sizes.
low_iter = ok[ok["iters"] < 10]
if not low_iter.empty:
    print("\nSolvers converging in <10 iterations (likely multigrid):")
    for _, r in low_iter.iterrows():
        print(f"  {r['solver']:10s}  {int(r['iters'])} iters  {r['t']:.2f}s")
    print("  These scale O(N) and will overtake PCG at larger grids.")

print(f"\nBest solver at 64³ by wall time: {best_solver}")


### 9b. Scaling validation — does the multigrid preconditioner break super-linear PCG?

§3b showed plain PCG scales as `t(N) ~ N^1.76` — that's the root cause of
slow large-grid solves, not any Python/binding overhead. A multigrid
preconditioner should restore near-`O(N)` scaling. The issue #256
acceptance target is **p < 1.2** at 64³ → 128³.

Fits `t = a·N^p` across 32³ / 64³ / 96³ / 128³ for three representative
choices: plain PCG, PCG+PFMG (Krylov + multigrid preconditioner), and MLMG
(AMReX-native geometric multigrid). Uses a uniform phase-0 block so each
solve is cheap and always percolates.


In [ ]:
scaling_sizes = [32, 64, 96, 128]
scaling_combos = [
    ("pcg",      {"solver": "pcg"}),
    ("pcg+pfmg", {"solver": "pcg", "preconditioner": "pfmg"}),
    ("mlmg",     {"solver": "mlmg"}),
]

scaling_records = []
for label, kw in scaling_combos:
    for N in scaling_sizes:
        arr = np.zeros((N, N, N), dtype=np.int32)  # uniform phase 0 — always percolates
        try:
            t0 = time.perf_counter()
            res = oi.tortuosity(arr, phase=0, direction="z",
                                max_grid_size=min(32, N), verbose=0, **kw)
            dt = time.perf_counter() - t0
            scaling_records.append({"solver": label, "N": N, "t": dt,
                                    "iters": int(res.iterations),
                                    "ok": bool(res.solver_converged)})
            print(f"  {label:10s}  {N:3d}^3  t={dt:7.3f}s  iters={res.iterations:4d}")
        except TypeError as e:
            # Wheel predates the preconditioner kwarg — skip only those rows.
            msg = str(e)
            if ("preconditioner" in msg) and ("preconditioner" in kw):
                print(f"  {label:10s}  {N:3d}^3  SKIP — wheel predates preconditioner kwarg")
                scaling_records.append({"solver": label, "N": N, "t": np.nan,
                                        "iters": 0, "ok": False})
                continue
            raise
        except Exception as e:
            scaling_records.append({"solver": label, "N": N, "t": np.nan,
                                    "iters": 0, "ok": False})
            print(f"  {label:10s}  {N:3d}^3  FAILED — {e}")

df_scale = pd.DataFrame(scaling_records)
df_scale


In [ ]:
# Fit t = a · N^p in log-log per solver, then overlay the curves with an
# O(N) reference line and the O(N^1.76) baseline from §3b.
fig, ax = plt.subplots(figsize=(9, 5.5))
palette = {"pcg": "#e74c3c", "pcg+pfmg": "#2ecc71", "mlmg": "#8e44ad"}
fit_results = {}
for label, _ in scaling_combos:
    sub = df_scale[(df_scale["solver"] == label) & df_scale["ok"]].dropna(subset=["t"])
    if len(sub) < 2:
        fit_results[label] = np.nan
        continue
    p, loga = np.polyfit(np.log(sub["N"].astype(float)), np.log(sub["t"].astype(float)), 1)
    fit_results[label] = p
    ax.loglog(sub["N"], sub["t"], "o-", color=palette.get(label, "#333333"),
              markersize=8, linewidth=2, label=f"{label}  (p={p:.2f})")

# Reference lines — anchored at the first successful measurement
anchor = df_scale[df_scale["ok"]].dropna(subset=["t"]).sort_values("N")
if not anchor.empty:
    N_anchor = float(anchor.iloc[0]["N"])
    t_anchor = float(anchor.iloc[0]["t"])
    Ns = np.array(scaling_sizes, dtype=float)
    ax.loglog(Ns, t_anchor * (Ns / N_anchor) ** 1.0,
              "k--", alpha=0.4, linewidth=1, label="O(N) reference")
    ax.loglog(Ns, t_anchor * (Ns / N_anchor) ** 1.76,
              "r:", alpha=0.5, linewidth=1, label="O(N^1.76) (§3b baseline)")

ax.set_xlabel("Grid size N  (domain = N^3)")
ax.set_ylabel("Wall time (s)")
ax.set_title("Scaling validation — multigrid vs. plain Krylov",
             fontweight="bold")
ax.legend(loc="upper left", framealpha=0.9)
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

# Verdict against acceptance target (issue #256: p < 1.2)
print("\nScaling exponents vs. acceptance target p < 1.2:")
for label, p in fit_results.items():
    if np.isnan(p):
        print(f"  {label:10s}  n/a  (insufficient data)")
    elif p < 1.2:
        print(f"  {label:10s}  p = {p:.2f}  ✓ near-linear — target met")
    else:
        print(f"  {label:10s}  p = {p:.2f}  ✗ super-linear — target NOT met")


## 10. `max_grid_size` tuning

Small boxes → better CPU cache + MPI parallelism. Large boxes → better
GPU kernel saturation. Sweep a few values with the best solver.


In [ ]:
grid_sizes = [8, 16, 32, 64]
grid_rows = []
for gs in grid_sizes:
    t0 = time.perf_counter()
    res = oi.tortuosity(data_medium, phase=0, direction="z",
                        solver=best_solver, max_grid_size=gs, verbose=0)
    dt = time.perf_counter() - t0
    grid_rows.append({"max_grid_size": gs, "t": dt, "tau": res.tortuosity})
    print(f"  max_grid_size={gs:3d}  t={dt:.2f}s  tau={res.tortuosity:.4f}")

df_grid = pd.DataFrame(grid_rows)
best_gs = int(df_grid.loc[df_grid["t"].idxmin(), "max_grid_size"])

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(df_grid["max_grid_size"], df_grid["t"], "o-", linewidth=2, markersize=9)
ax.set_xlabel("max_grid_size")
ax.set_ylabel("Wall time (s)")
ax.set_title(f"Grid sweep — 64³, solver={best_solver}", fontweight="bold")
plt.tight_layout()
plt.show()
print(f"Optimal max_grid_size: {best_gs}")


## 11. Native-binary comparison *(optional)*

Run the native `openimpala` executable on the same data (via a dumped
TIFF stack) and compare to the Python-binding call. The delta is
binding/orchestration overhead, nothing else.

Skip this if you don't have the native binary built — §3–§5 already
expose the Python-side story.


In [ ]:
native = shutil.which("openimpala")
if not native:
    print("No native openimpala binary on PATH — skipping comparison.")
else:
    import tifffile, tempfile, textwrap, os
    tmp = tempfile.mkdtemp()
    tif_path = os.path.join(tmp, "data.tif")
    tifffile.imwrite(tif_path, data_medium.astype(np.uint8))
    inputs_path = os.path.join(tmp, "inputs")
    with open(inputs_path, "w") as f:
        f.write(textwrap.dedent(f"""
            filename = {tif_path}
            phase_id = 0
            direction = Z
            solver_type = {best_solver}
            box_size = {best_gs}
            hypre.maxiter = 1000
            hypre.eps = 1e-9
            physics.type = diffusion
            verbose = 0
        """).strip())

    t0 = time.perf_counter()
    res = subprocess.run([native, inputs_path], capture_output=True, text=True, timeout=600)
    t_native = time.perf_counter() - t0

    # The native binary exits non-zero on input/parse errors but still returns
    # quickly — don't compare that 0.1s "win" to a real Python solve.
    if res.returncode != 0:
        print("=" * 70)
        print(f"  ⚠  Native binary exited with code {res.returncode} in {t_native:.2f}s")
        print("=" * 70)
        print("  It did NOT actually solve — skipping the comparison.")
        print()
        print("  stderr tail:")
        for line in (res.stderr or "").strip().splitlines()[-10:]:
            print(f"    {line}")
        if res.stdout:
            print("  stdout tail:")
            for line in res.stdout.strip().splitlines()[-5:]:
                print(f"    {line}")
        print("=" * 70)
        print("  Likely cause: the native binary on PATH expects a different")
        print("  inputs-file schema than what this cell writes (field names or")
        print("  types differ between versions). Fix the inputs file to match,")
        print("  or ignore this comparison — §3/§4 already prove the bindings")
        print("  add negligible overhead.")
    else:
        t0 = time.perf_counter()
        oi.tortuosity(data_medium, phase=0, direction="z",
                      solver=best_solver, max_grid_size=best_gs, verbose=0)
        t_py = time.perf_counter() - t0

        print(f"Native binary:    {t_native:.2f}s")
        print(f"Python binding:   {t_py:.2f}s")
        print(f"Binding overhead: {t_py - t_native:+.2f}s "
              f"({100*(t_py-t_native)/max(t_native,1e-9):+.0f}%)")


## 12. Summary & HPC recommendations


In [ ]:
# Collect headline numbers from the diagnostics above
stage_total = stages["_total"]
stage_top = max((k for k in stages if not k.startswith("_")), key=lambda k: stages[k])
fixed_overhead = a_fit
naive_mean = float(np.mean(naive))
amort_mean = float(np.mean(amort))

print("=" * 64)
print("  DIAGNOSIS")
print("=" * 64)
print(f"  Build backend:                  {backend.upper()}"
      + (f"   (host GPU: {gpu_name})" if gpu_name else ""))
print(f"  Single-call bottleneck:         {stage_top}")
print(f"                                  ({100*stages[stage_top]/stage_total:.1f}% of {stage_total:.2f}s)")
print(f"  Compute scaling:                O(N^{p_fit:.2f})"
      + ("   ⚠ super-linear" if p_fit > 1.2 else ""))
print(f"  Fixed per-call overhead (fit):  {fixed_overhead:.2f}s"
      + ("   (no meaningful fixed cost)" if not overhead_meaningful else ""))
print(f"  Per-call rebuild tax:           {naive_mean - amort_mean:.2f}s "
      f"({100*(naive_mean-amort_mean)/max(naive_mean,1e-9):.0f}% of naive call)")
print(f"  Best solver:                    {best_solver}")
print(f"  Best max_grid_size:             {best_gs}")
print("=" * 64)

# Headline recommendation — the #1 thing to change given what we found
print("\n  TOP RECOMMENDATION")
print("  " + "-" * 62)
if gpu_name and backend == "cpu":
    print(f"  You have a {gpu_name} sitting idle. The pip wheel is CPU-only.")
    print("  Rebuild openimpala with -DAMReX_GPU_BACKEND=CUDA — expect 10-50x.")
    print("  Everything below is secondary until this is addressed.")
elif p_fit > 1.2:
    print(f"  Solver is scaling O(N^{p_fit:.2f}), not O(N). PCG iterations grow")
    print("  with problem size. Switch to a multigrid preconditioner (SMG/PFMG")
    print("  via HYPRE, or MLMG) to restore linear scaling at large grids.")
else:
    print(f"  Solver dominates; best found is {best_solver} with max_grid_size={best_gs}.")
    print("  No free lunch here — further speedup needs algorithmic changes.")

# If the rebuild tax is large, recommend the amortized path
if naive_mean > 2 * amort_mean:
    print("\n⚠ Per-call setup dominates. For sweeps, build VoxelImage once:")
    print("    img = openimpala.facade._numpy_to_voxelimage(arr, max_grid_size)")
    print("    s   = openimpala._core.TortuosityHypre(img, vf, 0, d, st, '.', 0, 1, 0, False)")
    print("    s.value()")
    print("  …instead of calling oi.tortuosity(numpy, ...) each time.")


In [ ]:
inputs_template = f"""# Recommended OpenImpala inputs (generated by profiling notebook)
filename = your_image.tif
phase_id = 0
direction = ALL
solver_type = {best_solver}
box_size = {best_gs}
calculation_method = flow_through
hypre.maxiter = 1000
hypre.eps = 1e-9
physics.type = diffusion
verbose = 1
"""
print(inputs_template)


## Cleanup


In [ ]:
try:
    from openimpala._core import amrex_initialized
    if amrex_initialized():
        session.__exit__(None, None, None)
        print("Session closed.")
    else:
        print("Session already finalized.")
except Exception:
    print("Cleanup done.")
